# 02 · Model selection (run on LinuxONE)

Same data, same validation and same decision rule for every candidate.

- **Data:** `data/features.csv` (one row per city block, built by `scripts/prepare_features.py`).
- **Validation:** 5-fold cross-validation grouped by neighborhood (AGEB): models are always scored on neighborhoods they never saw.
- **Main metric:** hit rate in the top 10%: of the blocks a model flags as highest risk, the share that really are >= 25% vacant.
- **Also:** share of vacant homes captured in that top 10%, AUC, stability across folds, training and scoring time, explanation per block.
- **Decision rule:** best mean hit rate wins; models within one fold standard deviation of the best are a tie, broken by explainability and then scoring speed.

Uses only libraries in the LinuxONE Jupyter image (pandas, numpy, scikit-learn, XGBoost) plus Snap ML if installed (`pip install snapml`). XGBoost and Snap ML are skipped if they cannot load.

In [3]:
# On LinuxONE, install the extras the image does not have (Snap ML). Failure is fine: Snap ML is optional.
import platform, subprocess, sys
if platform.machine() == "s390x":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements-linuxone.txt"], check=False)


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import platform
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (HistGradientBoostingClassifier, HistGradientBoostingRegressor,
                              RandomForestClassifier, RandomForestRegressor)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
HIGH = 25  # % vacant that counts as a high-vacancy block
TOP = 0.10  # share of blocks an alert list would flag

libs = {}
try:
    import xgboost as xgb
    libs["xgboost"] = xgb.__version__
except Exception as e:  # on macOS it can fail with a missing OpenMP library, not only ImportError
    print("XGBoost skipped:", type(e).__name__)
try:
    import snapml
    libs["snapml"] = snapml.__version__
except Exception as e:
    print("Snap ML skipped:", type(e).__name__)
print(platform.machine(), "| Python", platform.python_version(), "|", libs)

Snap ML skipped: AttributeError
s390x | Python 3.12.15 | {'xgboost': '3.4.1'}


In [5]:
d = pd.read_csv("../data/features.csv")
ID_COLS = ["CVEGEO", "AGEB_KEY", "TVIVPAR", "vacant_masked", "target_pct_vacant", "lon", "lat"]
CRIME_VARIANTS = {"crime percentile": ["crimes_500m_pct"], "crime raw count": ["crimes_500m"], "no crime": []}
BASE = [c for c in d.columns if c not in ID_COLS + ["crimes_500m", "crimes_500m_pct"]]
FEATURES = BASE + CRIME_VARIANTS["crime percentile"]

y = d.target_pct_vacant.to_numpy()
high = (y >= HIGH).astype(int)
groups = d.AGEB_KEY.to_numpy()
vacant_homes = (d.target_pct_vacant * d.TVIVPAR / 100).to_numpy()
print(len(d), "blocks |", len(FEATURES), "features | high-vacancy base rate:", round(high.mean(), 3))

16452 blocks | 26 features | high-vacancy base rate: 0.113


## Candidates
Each entry: regressor factory, classifier factory (for the target variant), explanation per block.

In [6]:
def imputed(model, scale=False):
    steps = [SimpleImputer(strategy="median")] + ([StandardScaler()] if scale else []) + [model]
    return make_pipeline(*steps)

CANDIDATES = {
    "Baseline (median)": (lambda: DummyRegressor(strategy="median"), None, "n/a"),
    "Ridge": (lambda: imputed(RidgeCV(alphas=np.logspace(-2, 3, 12)), scale=True),
              lambda: imputed(LogisticRegression(max_iter=2000), scale=True), "yes"),
    "Random Forest": (lambda: imputed(RandomForestRegressor(300, min_samples_leaf=5, n_jobs=-1, random_state=0)),
                      lambda: imputed(RandomForestClassifier(300, min_samples_leaf=5, n_jobs=-1, random_state=0)), "partial"),
    "Gradient Boosting (sklearn)": (
        lambda: HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, min_samples_leaf=40, random_state=0),
        lambda: HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, min_samples_leaf=40, random_state=0), "partial"),
}
if "xgboost" in libs:
    xgb_params = dict(n_estimators=400, learning_rate=0.05, max_depth=6, subsample=0.8,
                      colsample_bytree=0.8, tree_method="hist", n_jobs=-1, random_state=0)
    CANDIDATES["XGBoost"] = (lambda: xgb.XGBRegressor(**xgb_params),
                             lambda: xgb.XGBClassifier(**xgb_params), "yes")  # pred_contribs gives per-block reasons
if "snapml" in libs:
    from snapml import BoostingMachineClassifier, BoostingMachineRegressor
    snap_params = dict(num_round=400, learning_rate=0.05, max_depth=6, random_state=0, n_jobs=4)
    CANDIDATES["Snap ML Boosting"] = (lambda: imputed(BoostingMachineRegressor(**snap_params)),
                                      lambda: imputed(BoostingMachineClassifier(**snap_params)), "partial")
list(CANDIDATES)

['Baseline (median)',
 'Ridge',
 'Random Forest',
 'Gradient Boosting (sklearn)',
 'XGBoost']

## Evaluation

In [7]:
def evaluate(make_model, features, classify=False):
    rows = []
    for fold, (tr, te) in enumerate(GroupKFold(5).split(d, y, groups)):
        model = make_model()
        target = high if classify else y
        t0 = time.perf_counter()
        model.fit(d.iloc[tr][features], target[tr])
        fit_s = time.perf_counter() - t0
        t0 = time.perf_counter()
        score = model.predict_proba(d.iloc[te][features])[:, 1] if classify else model.predict(d.iloc[te][features])
        score_s = time.perf_counter() - t0
        if np.ptp(score) > 0:
            flagged = score >= np.quantile(score, 1 - TOP)
        else:  # constant model (baseline): an alert list would be a random 10%
            flagged = np.random.default_rng(fold).random(len(te)) < TOP
        rows.append(dict(
            fold=fold,
            hit_rate_top10=high[te][flagged].mean(),
            vacant_captured_top10=vacant_homes[te][flagged].sum() / vacant_homes[te].sum(),
            auc=roc_auc_score(high[te], score) if np.ptp(score) > 0 else 0.5,
            fit_s=fit_s,
            score_ms_per_1k=score_s / len(te) * 1e6,
        ))
    return pd.DataFrame(rows)

def summarize(per_fold):
    s = per_fold.drop(columns="fold").agg(["mean", "std"]).T
    return s

results, per_fold = {}, {}
for name, (make_reg, _, explain) in CANDIDATES.items():
    per_fold[name] = evaluate(make_reg, FEATURES)
    s = summarize(per_fold[name])
    results[name] = {**s["mean"].round(3).to_dict(), "hit_rate_std": round(s.loc["hit_rate_top10", "std"], 3),
                     "explains_each_block": explain}
    print(f"{name}: hit rate {s.loc['hit_rate_top10', 'mean']:.3f}")

table = pd.DataFrame(results).T.sort_values("hit_rate_top10", ascending=False)
table

Baseline (median): hit rate 0.120
Ridge: hit rate 0.341
Random Forest: hit rate 0.462
Gradient Boosting (sklearn): hit rate 0.488
XGBoost: hit rate 0.481


,hit_rate_top10,vacant_captured_top10,auc,fit_s,score_ms_per_1k,hit_rate_std,explains_each_block
Gradient Boosting (sklearn),0.488,0.167,0.842,0.528,7.138,0.063,partial
XGBoost,0.481,0.169,0.843,1.371,6.723,0.072,yes
Random Forest,0.462,0.168,0.829,25.679,33.432,0.083,partial
Ridge,0.341,0.123,0.732,0.075,1.205,0.059,yes
Baseline (median),0.12,0.102,0.5,0.014,0.308,0.031,n/a


## Decision

In [8]:
EXPLAIN_RANK = {"yes": 0, "partial": 1, "n/a": 2}
best = table.iloc[0]
tied = table[table.hit_rate_top10 >= best.hit_rate_top10 - best.hit_rate_std].copy()
tied["explain_rank"] = tied.explains_each_block.map(EXPLAIN_RANK)
winner = tied.sort_values(["explain_rank", "score_ms_per_1k"]).index[0]
print("Tied with the best:", list(tied.index))
print("Winner:", winner)

Tied with the best: ['Gradient Boosting (sklearn)', 'XGBoost', 'Random Forest']
Winner: XGBoost


## Variants with the winner
Target: regression on % vacant vs classifier on >= 25% vacant. Crime: city percentile vs raw count vs none.

In [9]:
make_reg, make_clf, _ = CANDIDATES[winner]
variants = {}
for crime, cols in CRIME_VARIANTS.items():
    variants[f"regression, {crime}"] = summarize(evaluate(make_reg, BASE + cols))["mean"]
if make_clf:
    variants["classifier >=25%, crime percentile"] = summarize(evaluate(make_clf, FEATURES, classify=True))["mean"]
variant_table = pd.DataFrame(variants).T.round(3).sort_values("hit_rate_top10", ascending=False)
variant_table

,hit_rate_top10,vacant_captured_top10,auc,fit_s,score_ms_per_1k
"regression, no crime",0.491,0.171,0.846,1.513,6.532
"regression, crime percentile",0.481,0.169,0.843,1.375,6.278
"regression, crime raw count",0.481,0.169,0.843,1.413,6.300
"classifier >=25%, crime percentile",0.478,0.154,0.841,1.223,6.306


In [10]:
meta = f"{platform.machine()}, Python {platform.python_version()}, {libs}"
table.assign(platform=meta).to_csv("../results/model_selection.csv")
variant_table.assign(model=winner, platform=meta).to_csv("../results/model_variants.csv")
print("Saved results/ for the README:", meta)

Saved results/ for the README: s390x, Python 3.12.15, {'xgboost': '3.4.1'}
